In [1]:
import importlib

import src.preprocessing
import src.data_loader

importlib.reload(src.preprocessing)
importlib.reload(src.data_loader)

from src.data_loader import load_data

df = load_data()

print("DATASET SHAPE:", df.shape)

print("\nCOLUMNS:")
print(df.columns.tolist())

DATASET SHAPE: (8500, 25)

COLUMNS:
['detection_id', 'timestamp', 'camera_id', 'location', 'plate_number', 'registration_state', 'registration_lga', 'plate_type', 'car_brand', 'vehicle_category', 'detection_confidence', 'ocr_confidence', 'plate_type_confidence', 'image_path', 'vehicle_hash', 'processing_status', 'data_source', 'registration_admin_area', 'registration_admin_type', 'date', 'hour', 'day_name', 'day_of_week', 'day_type', 'year_month']


In [2]:
display(
    df[
        [
            "registration_state",
            "registration_admin_area",
            "registration_admin_type"
        ]
    ].drop_duplicates().sort_values(
        ["registration_state", "registration_admin_area"],
        na_position="last"
    )
)

,registration_state,registration_admin_area,registration_admin_type
5,Edo,Oredo,LGA
14,Enugu,Enugu North,LGA
11,FCT,Abuja Municipal,Area Council
41,FCT,Gwagwalada,Area Council
10,Kano,Kano Municipal,LGA
1,Kwara,Ilorin West,LGA
6,Lagos,Badagry,LGA
15,Lagos,Epe,LGA
0,Lagos,Ikeja,LGA
9,Lagos,Lagos Island,LGA


In [ ]:
# checking the loaded preprocessing.py loaded 

import inspect
import src.preprocessing

print(inspect.getsource(src.preprocessing.preprocess_dataset))

def preprocess_dataset(df):
    """Prepare validated detection records for analysis."""

    df = df.copy()

    # Convert timestamps to datetime.
    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        errors="coerce"
    )

    if df["timestamp"].isna().any():
        raise ValueError("Invalid or missing timestamps detected.")

    # The CSV stores local Lagos wall-clock time.
    if df["timestamp"].dt.tz is None:
        df["timestamp"] = df["timestamp"].dt.tz_localize(
            "Africa/Lagos"
        )
    else:
        df["timestamp"] = df["timestamp"].dt.tz_convert(
            "Africa/Lagos"
        )

    # Convert confidence columns to numeric.
    confidence_columns = [
        "detection_confidence",
        "ocr_confidence",
        "plate_type_confidence",
    ]

    for column in confidence_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

    # ----------------------------------------------------

In [4]:
print("ADMINISTRATIVE AREA AND TYPE:")
display(
    df[
        [
            "registration_state",
            "registration_admin_area",
            "registration_admin_type"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["registration_state", "registration_admin_area"],
        na_position="last"
    )
)

print("\nADMINISTRATIVE TYPE COUNTS:")
print(df["registration_admin_type"].value_counts(dropna=False))

print("\nMISSING ADMINISTRATIVE AREAS:")
print(df["registration_admin_area"].isna().sum())

ADMINISTRATIVE AREA AND TYPE:


,registration_state,registration_admin_area,registration_admin_type
5,Edo,Oredo,LGA
14,Enugu,Enugu North,LGA
11,FCT,Abuja Municipal,Area Council
41,FCT,Gwagwalada,Area Council
10,Kano,Kano Municipal,LGA
1,Kwara,Ilorin West,LGA
6,Lagos,Badagry,LGA
15,Lagos,Epe,LGA
0,Lagos,Ikeja,LGA
9,Lagos,Lagos Island,LGA



ADMINISTRATIVE TYPE COUNTS:
registration_admin_type
LGA             6796
Area Council     982
NaN              722
Name: count, dtype: int64

MISSING ADMINISTRATIVE AREAS:
722
